In [ ]:
%pip install vaderSentiment

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import html
import warnings
from scipy import stats
from scipy.stats import mannwhitneyu, ttest_ind
import matplotlib.pyplot as plt
import seaborn as sns
from textstat import flesch_kincaid_grade, gunning_fog
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.util import ngrams

warnings.filterwarnings('ignore')

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [3]:
CSV_PATH = 'recorded-data-final.csv'
QSF_PATH = 'recorded-data-final.qsf'
METRIC_NAMES = {
    'UND': 'Understandability',
    'TRU': 'Trustworthiness',
    'INS': 'Insightfulness',
    'SAT': 'Satisfaction',
    'CON': 'Confidence',
    'CVN': 'Convincingness',
    'COM': 'Communicability',
    'USB': 'Usability'
}
CRPS_QUESTION_MAP = {
    'Q135': 'UND', 'Q136': 'UND', 'Q138': 'UND', 'Q140': 'UND',
    'Q141': 'TRU', 'Q142': 'TRU', 'Q143': 'TRU', 'Q144': 'TRU',
    'Q145': 'INS', 'Q146': 'INS', 'Q147': 'INS', 'Q148': 'INS',
    'Q149': 'SAT', 'Q150': 'SAT', 'Q151': 'SAT', 'Q152': 'SAT',
    'Q153': 'CON', 'Q155': 'CON', 'Q154': 'CON', 'Q156': 'CON',
    'Q157': 'CVN', 'Q158': 'CVN', 'Q159': 'CVN', 'Q160': 'CVN',
    'Q161': 'COM', 'Q162': 'COM', 'Q163': 'COM', 'Q164': 'COM',
    'Q165': 'USB', 'Q166': 'USB', 'Q167': 'USB', 'Q168': 'USB'
}
NCRPS_QUESTION_MAP = {
    'NPA_1': 'UND', 'Q114': 'TRU', 'Q116': 'INS', 'Q117': 'SAT',
    'Q118': 'CON', 'Q119': 'CVN', 'Q120': 'COM', 'Q121': 'USB',
    'Q175': 'UND', 'Q176': 'UND', 'Q177': 'UND',
    'Q178': 'TRU', 'Q179': 'TRU', 'Q180': 'TRU',
    'Q181': 'INS', 'Q182': 'INS', 'Q183': 'INS',
    'Q184': 'SAT', 'Q185': 'SAT', 'Q186': 'SAT',
    'Q187': 'CON', 'Q188': 'CON', 'Q189': 'CON',
    'Q190': 'CVN', 'Q191': 'CVN', 'Q192': 'CVN',
    'Q194': 'COM', 'Q195': 'COM', 'Q196': 'COM',
    'Q197': 'USB', 'Q199': 'USB', 'Q201': 'USB'
}
MODEL_NAMES = {'1': 'Gemma 3', '2': 'DeepSeek R1', '3': 'Gemini 2.5'}
METRICS = ['UND', 'TRU', 'INS', 'SAT', 'CON', 'CVN', 'COM', 'USB']

In [4]:
NON_ANSWERS = {
    'unsure', 'prefer not to answer', 'i am not sure', "i don't know",
    'idk', 'n/a', 'na', 'not applicable', 'skip', 'blank'
}
LIKERT_WORDS = ('agree', 'disagree', 'neutral', 'strongly', 'slightly')

def _looks_like_nonanswer(s):
    s_low = s.lower().strip()
    return any(tok in s_low for tok in NON_ANSWERS)

def _extract_digit_candidates(s):
    cands = []
    cands += re.findall(r'\((\s*[1-5]\s*)\)', s)
    m = re.match(r'^\s*([1-5])\s*(?:[-–—]|\b)', s)
    if m:
        cands.append(m.group(1))
    if any(w in s.lower() for w in LIKERT_WORDS):
        cands += re.findall(r'(?<!\d)([1-5])(?!\d)', s)
    cands = [re.sub(r'\s+', '', d) for d in cands]
    cands = [d for d in cands if d in {'1', '2', '3', '4', '5'}]
    seen, uniq = set(), []
    for d in cands:
        if d not in seen:
            seen.add(d)
            uniq.append(d)
    return uniq

def likert(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val) if 1 <= float(val) <= 5 else np.nan

    s = str(val).strip()
    if s.startswith('{"ImportId'):
        return np.nan
    if _looks_like_nonanswer(s):
        return np.nan

    try:
        first = s.split(' - ')[0].strip()
        x = float(first)
        if 1 <= x <= 5:
            return x
    except:
        pass

    cands = _extract_digit_candidates(s)
    if len(cands) == 1:
        return float(cands[0])

    if re.fullmatch(r'[1-5]', s):
        return float(s)

    return np.nan

def to_cohort(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s.startswith('yes') or s in {'y', '1', 'true', 'crps', 'crp'}:
        return 'CRP'
    if s.startswith('no') or s in {'n', '0', 'false', 'ncrps', 'ncrp'}:
        return 'NCRP'
    return np.nan

In [5]:
def normalize_explanation_text(text):

    if pd.isna(text) or text == "":
        return ""
    
    text = str(text)
    text = html.unescape(text)
    text = re.sub(r'<br\s*/?>', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'<p[^>]*>', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'</p>', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    return text


In [6]:

_explanation_cache = {}

def get_explanation_for_loop_question(qsf_data, loop_num, question_id, cohort):
    global _explanation_cache

    cache_key = f"{loop_num}_{question_id}_{cohort}"
    if cache_key in _explanation_cache:
        return _explanation_cache[cache_key]

    survey_elements = qsf_data.get('SurveyElements', [])
    blocks = [e for e in survey_elements if e.get('Element') == 'BL']

    explanations = []

    for block in blocks:
        payload = block.get('Payload', {})
        for block_num, block_data in payload.items():
            if isinstance(block_data, dict):
                options = block_data.get('Options', {})
                if options:
                    looping = options.get('LoopingOptions', {})
                    if looping:
                        static = looping.get('Static', {})
                        if static and str(loop_num) in static:
                            loop_data = static[str(loop_num)]

                            for field_key in ['2', '3', '4', '5']:
                                if field_key in loop_data:
                                    exp_text = loop_data[field_key]
                                    if isinstance(exp_text, str) and len(exp_text) > 100:
                                        explanations.append(exp_text)

                            if explanations:
                                break

        if explanations:
            break

    if cohort == 'CRP' or cohort == 'CRPS':
        qmap = CRPS_QUESTION_MAP
    else:
        qmap = NCRPS_QUESTION_MAP

    metric = qmap.get(question_id, None)
    if metric:
        metric_questions = [q for q, m in qmap.items() if m == metric]
        if question_id in metric_questions:
            idx = metric_questions.index(question_id)
            if idx < len(explanations):
                explanation = explanations[idx]
                _explanation_cache[cache_key] = explanation
                return explanation

    return None

In [ ]:
df = pd.read_csv(CSV_PATH, skiprows=[1])
print(f"Loaded {len(df)} survey responses")

with open(QSF_PATH, 'r') as f:
    qsf_content = f.read()
qsf_data = json.loads(qsf_content)

df['cohort'] = df['Q92'].apply(to_cohort)
df['participant_id'] = df.get('ResponseId', pd.RangeIndex(len(df)))

print(f"\nCohort counts:")
print(df['cohort'].value_counts(dropna=False))

In [ ]:
all_ratings = []

for _, row in df.iterrows():
    pid = row['participant_id']
    cohort = row['cohort']

    if pd.isna(cohort):
        continue

    if cohort == 'CRPS':
        cohort = 'CRP'
    elif cohort == 'NCRPS':
        cohort = 'NCRP'

    qmap = CRPS_QUESTION_MAP if cohort == 'CRP' else NCRPS_QUESTION_MAP

    for loop_num in ['1', '2', '3']:
        model_name = MODEL_NAMES.get(loop_num, f'Model_{loop_num}')

        for question_id, metric in qmap.items():
            col = f"{loop_num}_{question_id}"

            if col in df.columns:
                rating = likert(row[col])

                if pd.notna(rating) and 1 <= rating <= 5:
                    explanation_text = get_explanation_for_loop_question(
                        qsf_data, loop_num, question_id, cohort
                    )

                    if explanation_text:
                        normalized_text = normalize_explanation_text(explanation_text)
                        
                        all_ratings.append({
                            'participant_id': pid,
                            'cohort': cohort,
                            'model': model_name,
                            'metric': metric,
                            'metric_name': METRIC_NAMES.get(metric, metric),
                            'loop_num': loop_num,
                            'question_id': question_id,
                            'rating': rating,
                            'explanation_text': explanation_text,
                            'normalized_text': normalized_text
                        })

ratings_df = pd.DataFrame(all_ratings)
print(f"Collected {len(ratings_df)} ratings with explanations")
print(f"Unique normalized explanations: {ratings_df['normalized_text'].nunique()}")
print(f"\nRatings per metric:")
print(ratings_df['metric'].value_counts())

In [ ]:
und_data = ratings_df[ratings_df['metric'] == 'UND'].copy()
und_data['percentile'] = und_data['rating'].rank(pct=True) * 100

In [ ]:
for metric in METRICS:
    metric_df = ratings_df[ratings_df['metric'] == metric].copy()
    
    if len(metric_df) == 0:
        continue
    
    metric_df['percentile'] = metric_df['rating'].rank(pct=True, method='first') * 100
    metric_df['rating_group'] = 'middle'
    metric_df.loc[metric_df['percentile'] >= 75, 'rating_group'] = 'high'
    metric_df.loc[metric_df['percentile'] <= 25, 'rating_group'] = 'low'
    high_count = len(metric_df[metric_df['rating_group'] == 'high'])
    low_count = len(metric_df[metric_df['rating_group'] == 'low'])
    
    if high_count == 0:
        top_25_pct = max(1, int(len(metric_df) * 0.25))  
        if top_25_pct > 0:
            top_ratings = metric_df.nlargest(top_25_pct, 'rating')
            metric_df.loc[top_ratings.index, 'rating_group'] = 'high'
            metric_df.loc[top_ratings.index, 'percentile'] = 100
    
    if low_count == 0:
        bottom_25_pct = max(1, int(len(metric_df) * 0.25))  # At least 1
        if bottom_25_pct > 0:
            bottom_ratings = metric_df.nsmallest(bottom_25_pct, 'rating')
            metric_df.loc[bottom_ratings.index, 'rating_group'] = 'low'
            metric_df.loc[bottom_ratings.index, 'percentile'] = 0
    
    ratings_df.loc[ratings_df['metric'] == metric, 'percentile'] = metric_df['percentile']
    ratings_df.loc[ratings_df['metric'] == metric, 'rating_group'] = metric_df['rating_group']

for metric in METRICS:
    metric_df = ratings_df[ratings_df['metric'] == metric]
    if len(metric_df) > 0:
        print(f"\n{METRIC_NAMES[metric]} ({metric}):")
        print(metric_df['rating_group'].value_counts())


In [11]:
vader_analyzer = SentimentIntensityAnalyzer()
HEDGE_WORDS = ['may', 'might', 'could', 'possibly', 'perhaps', 'suggests', 
               'indicates', 'appears', 'seems', 'likely', 'probably', 'maybe']
CAUSAL_OVERCLAIMING = ['means', 'proves', 'shows that', 'demonstrates that', 
                       'confirms that', 'establishes that', 'determines that']
PRESCRIPTIVE_LANGUAGE = ['you should', 'try to', 'consider doing', 'you need to',
                        'you must', 'you ought to', 'it is recommended', 
                        'it is advisable', 'you might want to']
SHAP_FEATURES = ['fico', 'dti', 'ltv', 'mi_pct', 'debt', 'income', 'loan amount',
                 'credit score', 'debt-to-income', 'loan-to-value', 'mortgage insurance']
NETWORK_TERMS = ['connected', 'borrowers in your area', 'cluster', 'patterns',
                 'network', 'neighbors', 'similar borrowers', 'community',
                 'local', 'regional', 'geographic']

def extract_linguistic_features(text):
    if pd.isna(text) or text == "":
        return {
            'word_count': 0,
            'sentence_count': 0,
            'flesch_kincaid': np.nan,
            'gunning_fog': np.nan,
            'sentiment_polarity': 0.0,
            'sentiment_subjectivity': 0.0,
            'vader_compound': 0.0,
            'vader_pos': 0.0,
            'vader_neu': 0.0,
            'vader_neg': 0.0,
            'hedge_count': 0,
            'causal_overclaiming_count': 0,
            'prescriptive_count': 0,
            'numeric_count': 0,
            'shap_feature_count': 0,
            'network_term_count': 0,
            'unique_tokens': 0,
            'bigrams': [],
            'trigrams': []
        }
    
    words = word_tokenize(text)
    sentences = sent_tokenize(text)
    word_count = len(words)
    sentence_count = len(sentences)
    
    try:
        flesch_kincaid = flesch_kincaid_grade(text)
    except:
        flesch_kincaid = np.nan
    
    try:
        gunning_fog_score = gunning_fog(text)
    except:
        gunning_fog_score = np.nan
    
    try:
        blob = TextBlob(text)
        sentiment_polarity = blob.sentiment.polarity
        sentiment_subjectivity = blob.sentiment.subjectivity
    except:
        sentiment_polarity = 0.0
        sentiment_subjectivity = 0.0
    
    try:
        vader_scores = vader_analyzer.polarity_scores(text)
        vader_compound = vader_scores['compound']
        vader_pos = vader_scores['pos']
        vader_neu = vader_scores['neu']
        vader_neg = vader_scores['neg']
    except:
        vader_compound = 0.0
        vader_pos = 0.0
        vader_neu = 0.0
        vader_neg = 0.0
    
    text_lower = text.lower()
    hedge_count = sum(1 for word in HEDGE_WORDS if word in text_lower)
    causal_count = sum(1 for phrase in CAUSAL_OVERCLAIMING if phrase in text_lower)
    prescriptive_count = sum(1 for phrase in PRESCRIPTIVE_LANGUAGE if phrase in text_lower)
    numeric_count = len(re.findall(r'\d+', text))
    shap_count = sum(1 for feature in SHAP_FEATURES if feature in text_lower)
    network_count = sum(1 for term in NETWORK_TERMS if term in text_lower)
    unique_tokens = len(set(words))
    try:
        bigrams_list = list(ngrams(words, 2))
        trigrams_list = list(ngrams(words, 3))
    except:
        bigrams_list = []
        trigrams_list = []
    
    return {
        'word_count': word_count,
        'sentence_count': sentence_count,
        'flesch_kincaid': flesch_kincaid,
        'gunning_fog': gunning_fog_score,
        'sentiment_polarity': sentiment_polarity,
        'sentiment_subjectivity': sentiment_subjectivity,
        'vader_compound': vader_compound,
        'vader_pos': vader_pos,
        'vader_neu': vader_neu,
        'vader_neg': vader_neg,
        'hedge_count': hedge_count,
        'causal_overclaiming_count': causal_count,
        'prescriptive_count': prescriptive_count,
        'numeric_count': numeric_count,
        'shap_feature_count': shap_count,
        'network_term_count': network_count,
        'unique_tokens': unique_tokens,
        'bigrams': bigrams_list,
        'trigrams': trigrams_list
    }


In [ ]:
nltk.download("punkt_tab")

In [ ]:
feature_list = []

for idx, row in ratings_df.iterrows():
    features = extract_linguistic_features(row['explanation_text'])
    features['idx'] = idx
    feature_list.append(features)

features_df = pd.DataFrame(feature_list)
features_df.set_index('idx', inplace=True)
feature_columns = list(features_df.columns)
existing_columns = [col for col in feature_columns if col in ratings_df.columns]
if existing_columns:
    print(f"Dropping existing feature columns: {existing_columns}")
    ratings_df = ratings_df.drop(columns=existing_columns)

# Merge with ratings dataframe
ratings_df = ratings_df.join(features_df, how='left')

print(f"Extracted features for {len(ratings_df)} ratings")
print(f"\nFeature columns: {list(features_df.columns)}")
print(f"\nSample statistics:")
print(ratings_df[['word_count', 'sentence_count', 'flesch_kincaid', 'gunning_fog']].describe())


In [ ]:
FEATURE_COLUMNS = [
    'word_count', 'sentence_count', 'flesch_kincaid', 'gunning_fog',
    'sentiment_polarity', 'sentiment_subjectivity',
    'vader_compound', 'vader_pos', 'vader_neu', 'vader_neg',
    'hedge_count', 'causal_overclaiming_count', 'prescriptive_count',
    'numeric_count', 'shap_feature_count', 'network_term_count',
    'unique_tokens'
]

def compute_effect_size(high_values, low_values):
    high_clean = high_values[~np.isnan(high_values)]
    low_clean = low_values[~np.isnan(low_values)]
    
    if len(high_clean) < 2 or len(low_clean) < 2:
        return np.nan
    
    high_mean = np.mean(high_clean)
    low_mean = np.mean(low_clean)
    high_std = np.std(high_clean, ddof=1)
    low_std = np.std(low_clean, ddof=1)
    n_high = len(high_clean)
    n_low = len(low_clean)
    
    if n_high + n_low - 2 == 0:
        return np.nan
    
    pooled_std = np.sqrt(((n_high - 1) * high_std**2 + (n_low - 1) * low_std**2) / (n_high + n_low - 2))
    
    if pooled_std == 0:
        return np.nan
    
    cohens_d = (high_mean - low_mean) / pooled_std
    return cohens_d

all_results = []

for metric in METRICS:
    metric_df = ratings_df[ratings_df['metric'] == metric].copy()
    
    if len(metric_df) == 0:
        continue
    
    high_df = metric_df[metric_df['rating_group'] == 'high']
    low_df = metric_df[metric_df['rating_group'] == 'low']
    
    if len(high_df) == 0 or len(low_df) == 0:
        print(f"Warning: Insufficient data for {metric}")
        continue
    
    print(f"\nAnalyzing {METRIC_NAMES[metric]} ({metric})")
    print(f"High group: {len(high_df)} ratings")
    print(f"Low group: {len(low_df)} ratings")
    
    metric_results = []
    
    for feature in FEATURE_COLUMNS:
        high_values = high_df[feature].values
        low_values = low_df[feature].values
        high_clean = high_values[~np.isnan(high_values)]
        low_clean = low_values[~np.isnan(low_values)]
        
        if len(high_clean) < 3 or len(low_clean) < 3:
            continue
        
        high_mean = np.mean(high_clean)
        low_mean = np.mean(low_clean)
        high_std = np.std(high_clean, ddof=1)
        low_std = np.std(low_clean, ddof=1)
        
        try:
            u_stat, u_pvalue = mannwhitneyu(high_clean, low_clean, alternative='two-sided')
        except:
            u_stat, u_pvalue = np.nan, np.nan
        
        try:
            t_stat, t_pvalue = ttest_ind(high_clean, low_clean)
        except:
            t_stat, t_pvalue = np.nan, np.nan
        
        cohens_d = compute_effect_size(high_clean, low_clean)
        
        metric_results.append({
            'metric': metric,
            'metric_name': METRIC_NAMES[metric],
            'feature': feature,
            'high_mean': high_mean,
            'high_std': high_std,
            'high_n': len(high_clean),
            'low_mean': low_mean,
            'low_std': low_std,
            'low_n': len(low_clean),
            'mean_difference': high_mean - low_mean,
            'mannwhitney_u': u_stat,
            'mannwhitney_p': u_pvalue,
            't_statistic': t_stat,
            't_pvalue': t_pvalue,
            'cohens_d': cohens_d,
            'abs_cohens_d': abs(cohens_d) if not np.isnan(cohens_d) else np.nan
        })
    
    all_results.extend(metric_results)

results_df = pd.DataFrame(all_results)
print(f"\n\nTotal comparisons: {len(results_df)}")

In [ ]:
summary_table = results_df.pivot_table(
    index=['metric_name', 'feature'],
    values=['high_mean', 'low_mean', 'mean_difference', 'cohens_d', 'mannwhitney_p'],
    aggfunc='first'
).round(3)

summary_table['abs_cohens_d'] = summary_table['cohens_d'].abs()
summary_table = summary_table.sort_values(['metric_name', 'abs_cohens_d'], ascending=[True, False])
print(summary_table.to_string())

In [ ]:
metric_order = ['UND', 'TRU', 'INS', 'SAT', 'CON', 'CVN', 'COM', 'USB']
metric_mapping = {
    'Understandability': 'UND',
    'Trustworthiness': 'TRU',
    'Insightfulness': 'INS',
    'Satisfaction': 'SAT',
    'Confidence': 'CON',
    'Convincingness': 'CVN',
    'Communicability': 'COM',
    'Usability': 'USB'
}

results_df['metric_acronym'] = results_df['metric_name'].map(metric_mapping)
filtered_df = results_df[results_df['metric_acronym'].isin(metric_order)]
effect_size_pivot = filtered_df.pivot_table(
    index='feature',
    columns='metric_acronym',
    values='cohens_d',
    aggfunc='mean'
)

effect_size_pivot = effect_size_pivot.reindex(columns=metric_order)

plt.figure(figsize=(14, 10))

ax = sns.heatmap(
    effect_size_pivot,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    cbar_kws={'label': "Cohen's d (High - Low)"},
    linewidths=0.5,
    annot_kws={'size': 14}
)
ax.tick_params(axis="x", labelsize=14, rotation=0)
ax.tick_params(axis="y", labelsize=14)
plt.xlabel('Metric', fontsize=16)
plt.ylabel('Linguistic Feature', fontsize=16)
cbar = ax.collections[0].colorbar
cbar.set_label("Cohen's d (High - Low)", fontsize=16)
cbar.ax.tick_params(labelsize=16)
plt.savefig('heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
results_df.to_csv('text_analysis_results.csv', index=False)
print("Saved: text_analysis_results.csv")

summary_stats = ratings_df.groupby(['metric', 'rating_group'])[FEATURE_COLUMNS].agg(['mean', 'std']).round(3)
summary_stats.to_csv('text_analysis_summary_stats.csv')
print("Saved: text_analysis_summary_stats.csv")